# 04 — Customer Segmentation
**Pipeline:** RFM feature scaling → K-Means tuning (elbow + silhouette) → cluster stability check → segment profiling → classifier for inference → MLflow logging → artifact export.

> **Design note:** K-Means is unsupervised — there are no labels, so no train/test split is needed for clustering itself.  
> A `RandomForestClassifier` is trained *on top* of the cluster labels (supervised) so new customers can be scored without re-running K-Means on the full dataset.  
> That classifier uses a proper train/test split and is the artifact served at inference time.


## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
%matplotlib inline


/Users/lathiyahit/Documents/Customer Segmentation And Retention/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load features

In [2]:
df = pd.read_parquet("../data/processed/features.parquet")
print(f"Shape: {df.shape}")
df.head()


Shape: (4290, 20)


,CustomerID,Recency,Frequency,Monetary,Log_Monetary,Log_Frequency,AOV,SpendStd,PreferredDayOfWeek,WeekendRatio,PreferredHour,UniqueSKUs,TotalItems,RepeatSKURatio,AvgGap,StdGap,ReturnRate,CohortMonth,ActiveMonths,DaysSinceFirstPurchase
0,12347,2,7,4060.40,8.309283,2.079442,580.057143,351.455168,1,0.000000,14,102,2218,0.436464,60.333333,18.478817,0.000000,2010-12,7,365
1,12348,75,4,1186.68,7.079757,1.609438,296.670000,225.294269,3,0.095238,19,15,1468,0.285714,94.000000,70.149840,0.000000,2010-12,4,282
2,12349,19,1,1353.80,7.211409,0.693147,1353.800000,0.000000,0,0.000000,9,70,625,0.000000,0.000000,0.000000,0.000000,2011-11,1,0
3,12350,310,1,294.40,5.688330,0.693147,294.400000,0.000000,2,0.000000,16,16,196,0.000000,0.000000,0.000000,0.000000,2011-02,1,0
4,12352,36,7,1385.74,7.234711,2.079442,197.962857,81.916277,1,0.000000,14,57,526,0.259740,43.000000,68.419296,0.909091,2011-02,4,260


## 3. Prepare RFM matrix
> **Scaler note:** `StandardScaler` is fitted on the full dataset here because this is an unsupervised pipeline with no held-out labels.  
> In production, you would `fit` on a historical snapshot and `transform` new customer batches — the saved `scaler_rfm.joblib` artifact supports exactly that workflow.


In [3]:
rfm_cols = ["Recency", "Log_Frequency", "Log_Monetary"]
X_raw = df[rfm_cols].copy()

# NOTE: clustering happens in log-space (Log_Frequency, Log_Monetary) to reduce
# right-skew; segment profiles below are reported in original units for readability.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Save scaled values back into the dataframe
df["Recency_scaled"] = X_scaled[:, 0]
df["Log_Frequency_scaled"] = X_scaled[:, 1]
df["Log_Monetary_scaled"] = X_scaled[:, 2]
print("Scaled shape:", X_scaled.shape)


Scaled shape: (4290, 3)
